# Ollama에서 Hugging Face GGUF 모델 사용하기

Hugging Face에 공개된 GGUF 모델은 파일을 별도 폴더에 내려받고 `Modelfile`로 등록하지 않아도 `hf.co/{사용자}/{저장소}:{양자화}` 형식으로 Ollama에 가져올 수 있다. 이 노트북에서는 모델을 Ollama cache에 한 번 준비한 뒤 Python API와 LangChain에서 같은 model ID를 재사용한다.


## GGUF 포맷

[GGUF](https://huggingface.co/docs/hub/en/gguf)는 모델 가중치뿐 아니라 tokenizer와 실행에 필요한 metadata를 하나의 파일에 담는 포맷이다. llama.cpp와 Ollama 같은 경량 추론 엔진이 빠르게 읽을 수 있으며, 여러 양자화 방식을 지원한다.

- **단일 파일**: 가중치와 metadata를 함께 보관해 배포하기 쉽다.
- **양자화 지원**: 4비트·5비트처럼 낮은 정밀도로 저장해 파일 크기와 추론 메모리를 줄일 수 있다.
- **실행 엔진 호환**: llama.cpp 계열 도구와 Ollama에서 사용할 수 있다.

`Q5_K_M`은 5비트 계열 양자화 방식이다. 일반적으로 더 낮은 bit 수는 메모리를 줄이지만 원본 가중치와의 차이가 커질 수 있다.

## 선수 조건과 패키지 준비

RunPod에서 `01_ollama.ipynb`를 먼저 완료해 Ollama server가 실행 중이어야 한다. Python의 `ollama` package는 새 server를 만드는 도구가 아니라 같은 Pod의 `http://localhost:11434` server에 요청하는 client이다.

In [3]:
%pip install -U ollama langchain-ollama

import ollama
from langchain_ollama import ChatOllama


Note: you may need to restart the kernel to use updated packages.


## Hugging Face GGUF 모델 바로 사용하기

`heegyu/EEVE-Korean-Instruct-10.8B-v1.0-GGUF:latest` 저장소는 현재 Ollama의 Hugging Face manifest 처리에서 호환 오류가 발생한다. 따라서 같은 `yanolja/EEVE-Korean-Instruct-10.8B-v1.0` 기반의 `Q5_K_M` GGUF인 [SourPineapple 저장소](https://huggingface.co/SourPineapple/EEVE-Korean-Instruct-10.8B-v1.0-Q5_K_M-GGUF)를 사용한다.

`ollama pull`은 GGUF를 임의의 `/workspace` 폴더에 저장하는 명령이 아니다. Ollama가 관리하는 model cache에 최초 한 번 내려받아 이후 `generate()`와 `chat()`이 같은 model ID를 사용할 수 있게 한다. [Hugging Face의 Ollama 가이드](https://huggingface.co/docs/hub/en/ollama)는 저장소 뒤에 `:Q5_K_M`처럼 tag를 붙여 원하는 양자화를 선택하는 형식을 제공한다.

In [2]:
MODEL_ID = (
    'hf.co/SourPineapple/'
    'EEVE-Korean-Instruct-10.8B-v1.0-Q5_K_M-GGUF:Q5_K_M'
)


### Ollama cache에 모델 준비하기

Python client의 `pull()`에 model ID를 전달하면 Ollama server가 Hugging Face에서 GGUF를 내려받아 자체 cache로 관리한다. 최초 실행은 파일 크기만큼 시간이 필요하지만 이후 실행에서는 저장된 layer를 재사용한다.

In [4]:
import ollama

pull_result =ollama.pull(MODEL_ID)
print(pull_result)


status='success' completed=None total=None digest=None


## `ollama.generate()`로 한 번 생성하기

`generate()`는 하나의 prompt를 전달해 assistant 역할 구분 없이 텍스트를 생성한다. `model`과 `prompt`를 전달하며, 반환값의 `response`에 생성된 본문이 들어 있다.

In [5]:
response = ollama.generate(
    model =MODEL_ID,
    prompt = "Ollama가 뭐야?"
)
print(response['response'])

Olla ma는 핀란드어로 '여기 산다'라는 뜻입니다. 이는 사람이 사는 장소를 나타내거나 자신의 현재 위치를 설명할 때 사용됩니다.


## `ollama.chat()`으로 역할이 있는 대화하기

`chat()`은 `system`, `user`, `assistant` 역할이 있는 message 목록을 전달한다. 반환값의 `message.content`에서 assistant 답변을 꺼낸다.

In [9]:
chat_response = ollama.chat(
    model =MODEL_ID,
    messages =[
        {
            'role':'system',
            'content':'너는 현업 AI 엔지니어로 일하고 있고, 조언을 해주는 역할이야,관련 질문에 대해서 간략하게 2문단 이내로 부탁해  '
        },

        {
        'role':'user',
        'content':'신입 AI 엔지니어가 되려면 무엇을 준비해야 해?'
    }]
)
print(chat_response['message']['content'])

신입 AI 엔지니어로 성장하려면 다음과 같은 단계와 기술을 준비하는 것이 필수적입니다:

1. 강력한 수학 및 컴퓨터 과학 기초: 선형 대수, 미적분, 통계학, 컴퓨터 프로그래밍 같은 분야에서 탄탄한 기반을 닦으세요. 파이썬, 자바스크립트, C++와 같은 프로그래밍 언어에 익숙해지세요.

2. 인공지능에 대한 지식 습득: 머신러닝, 딥러닝, 자연어 처리, 컴퓨터 비전 등의 주제에 대해 공부하세요. 이 기술들이 어떻게 작동하는지, 다양한 AI 모델을 구현하고 훈련시키기 위해 어떤 알고리즘을 사용하는지 이해하세요.

3. 프로그래밍 실력 연마: Git, 버전 관리, 협업 도구와 같은 프로그래밍 도구 및 기술에 익숙해지세요. 다양한 프로그래밍 언어와 라이브러리를 사용한 프로젝트 포트폴리오를 구축하세요.

4. AI 도구와 프레임워크에 익숙해지기: TensorFlow, PyTorch, Scikit-learn, Keras와 같은 인기 있는 AI 도구와 프레임워크를 사용해보세요. 이 도구들을 사용하여 AI 모델을 만들고 훈련하며, 그 모델의 성능을 최적화하세요.

5. 문제 해결 능력 개발: 머신러닝과 딥러닝에서 흔히 마주치는 도전 과제를 해결할 수 있는 논리적이고 창의적으로 생각하는 능력을 개발하세요. 복잡한 문제를 작은 부분으로 나누고, 그것들을 해결한 다음, 다시 조립하는 방법을 이해하세요.

6. 협업 및 소통 능력 향상: AI 프로젝트는 종종 다학제 팀과 협력하는 것을 포함합니다. 동료들과 효과적으로 소통하고 협업할 수 있는 능력을 개발하세요.

7. 실제 AI 프로젝트에 참여하기: 인턴십, 멘토링 프로그램, 또는 오픈 소스 프로젝트에서 AI 엔지니어로 일하는 기회를 찾아보세요. 실제 프로젝트 경험은 AI 엔지니어의 일상 업무와 AI 분야에서 성공하기 위해 필요한 기술을 이해하는 데 도움이 될 것입니다.

8. 지속적인 학습: 인공지능 분야는 빠르게 발전하고 있습니다. 최신 연구, 트렌드, 기술에 대해 정보를 유지하여 경쟁력을 유지하고 최신 상태를 유지하세요.


## LangChain `ChatOllama`로 같은 모델 호출하기

`ChatOllama`는 같은 Ollama server와 model ID를 LangChain의 Chat Model 인터페이스로 감싼다. GGUF를 다시 내려받거나 다른 모델로 변환하는 과정과 관계가 없다.

In [10]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model = MODEL_ID,
    temperature = 0.2,
)
langchain_response = llm.invoke("대한민국 24절기에 대해서 간단히 설명해줘.")

print(langchain_response.content)



대한민국 24절기는 우리 조상들이 자연의 변화를 관찰하고 그에 따라 생활의 리듬을 맞추기 위해 정한 전통 절기입니다. 24절기는 1년 중 24개의 주요한 날들로 구성되어 있으며, 계절의 변화를 반영하고 있습니다. 각 절기는 음력 날짜에 따라 정해지며, 태양력과 약간 차이가 납니다.

24절기는 4계절(봄, 여름, 가을, 겨울)로 나뉘며, 각 계절에는 6개의 절기가 있습니다. 24절기의 이름과 그 의미에 대한 간략한 설명은 다음과 같습니다:

1. 입춘(立春): 봄의 시작을 알리는 절기로, 음력 1월 20일경입니다.
2. 우수(雨水): 봄비가 내리는 시기이며, 음력 2월 19일경입니다.
3. 경칩(驚蟄): 겨울잠을 자던 벌레들이 깨어나는 시기, 음력 2월 20일경입니다.
4. 청명(淸明): 봄의 시작을 알리는 절기로, 음력 3월 20일경입니다.
5. 곡우(穀雨): 봄비가 내려 곡식을 기르는 시기, 음력 4월 20일경입니다.
6. 입하(立夏): 여름의 시작을 알리는 절기로, 음력 5월 5일경입니다.

7. 소만(小滿): 여름이 시작되는 시기, 음력 5월 21일경입니다.
8. 망종(芒種): 보리를 수확하는 시기, 음력 6월 5일경입니다.
9. 하지(夏至): 여름의 시작을 알리는 절기로, 음력 6월 21일경입니다.
10. 소서(小暑): 여름이 시작되는 시기, 음력 7월 7일경입니다.
11. 대서(大暑): 여름이 시작되는 시기, 음력 7월 23일경입니다.
12. 입추(立秋): 가을의 시작을 알리는 절기로, 음력 8월 7일경입니다.

13. 처서(處暑): 여름이 끝나는 시기, 음력 8월 23일경입니다.
14. 백로(白露): 이슬이 맺히는 시기, 음력 9월 8일경입니다.
15. 추분(秋分): 가을의 시작을 알리는 절기로, 음력 9월 23일경입니다.
16. 한로(寒露): 추운 이슬이 내리는 시기, 음력 10월 8일경입니다.
17. 상강(霜降): 서리가 내리는 시기, 음력 10월 23일경입니다.

18. 입동(立冬): 겨울의 시작을 알리는 절기로, 음력 11월 7일경입니다.


## 선택 확장: 직접 `Modelfile`을 만드는 경우

두 번째 경로인 `GGUF 다운로드 → Modelfile 작성 → ollama create`는 model의 prompt template와 생성 parameter를 직접 바꿀 때만 사용한다. 기본 실습처럼 공개 GGUF를 그대로 실행할 때는 필요하지 않으므로 이 노트북의 필수 실행 단계에서 제외한다. 변환과 수동 등록 과정은 `hf_to_gguf.ipynb`에서 별도로 다룬다.

## 정리

- 기본 실습은 Hugging Face model ID를 Ollama cache에 준비하고 `ollama.generate()`로 호출한다.
- `/workspace`에 GGUF를 따로 내려받는 과정은 기본 실습에서 제외한다.
- `generate()`, `chat()`, `ChatOllama`는 같은 Ollama server와 같은 model을 서로 다른 입력 형식으로 호출한다.
- `Modelfile`은 공개 GGUF의 template나 parameter를 직접 바꿔야 할 때만 선택한다.